# 09 · Analysis: metrics, significance, files and diagrams

Plan section 6. Reads every run in `results/runs/` (v1, and v2 once it exists) and writes:

* `results/run_metrics.parquet` (`run_id, index, level, metric, value, subset, n, ci_lo, ci_hi`) and `results/question_metrics.parquet` (per question, for the paired tests);
* `results/units/metrics.{csv,xlsx}` and `results/win500/metrics.{csv,xlsx}`: **one table per index**, to be merged by hand later;
* `results/granularity.csv`: RQ7, windows vs units;
* `results/diagrams/<run_id>.md`: Mermaid pipeline diagrams of hand-picked runs.

**n = 92 answerable; 8 unanswerable excluded** in every metric table. Tables are sorted by section-level nDCG@10 for readability only: every metric is reported and read, and no run is selected as the winner.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from csrag import evaluate as ev, report as rp, runs
from csrag.corpus import Corpus
from csrag.retrievers.base import EXPANSION_METHODS, RANKED_METHODS

corpus = Corpus(DATA)
questions, problems = ev.load_questions(DATA / "eval" / "questions.jsonl", corpus)
qrels = ev.read_qrels(DATA / "eval" / "qrels.jsonl")
run_ids = runs.list_runs()
rt = ev.run_table(run_ids)
rt.groupby(["params_version", "index", "gate"]).size().rename("runs").to_frame().T

## Evaluate every run

In [ ]:
pq, rm, gates, costs = ev.evaluate_all(run_ids, questions, qrels, corpus, progress=tqdm)
gate_overall = ev.gate_loss_vs_gate_off(pq, rt)
RESULTS.mkdir(exist_ok=True)
rm.to_parquet(RESULTS / "run_metrics.parquet", index=False)
pq.to_parquet(RESULTS / "question_metrics.parquet", index=False)
# first_gold_rank is NaN when the gold is not retrieved, so it is the one metric with n < 92
scored = rm[(rm["subset"] == "all") & rm["level"].isin(["unit", "section", "group", "evidence"]) & (rm["metric"] != "first_gold_rank")]
assert (scored["n"] == 92).all(), "every scored metric row must have n = 92"
print(len(rm), "metric rows for", rm["run_id"].nunique(), "runs; n = 92 answerable, 8 unanswerable excluded")

## Significance (pre-declared comparisons only)

Paired randomisation tests (sign flip, 10,000 permutations) on every metric. Holm correction within each RQ, metric and index (and params version). RQ1–RQ6 are tested within each index with matched gate and budget; RQ7 compares the two indexes with the gate off, at section/group level and on evidence recall. Everything else is descriptive.

In [ ]:
qids = [q["question_id"] for q in questions if q["answerable"]]
sig = pd.concat([ev.significance(pq, rt, qids, index=i) for i in corpus.indexes()], ignore_index=True)
sig7 = ev.significance(pq, rt, qids, rq7=True)
key = (sig["level"] == "section") & (sig["metric"] == "ndcg@10")
sig[key][["rq", "a", "b", "index", "gate", "budget", "params_version", "mean_a", "mean_b", "diff", "diff_ci_lo",
          "diff_ci_hi", "p", "p_holm"]].round(4)

## Write the metrics files

In [ ]:
written = {i: rp.write_index_files(i, rm, pq, rt, gates, costs, gate_overall, sig, questions, corpus) for i in corpus.indexes()}
rp.write_granularity(sig7, RESULTS / "granularity.csv")
written

## Tables by research question

Key columns only; the CSV/XLSX files hold every metric.

In [ ]:
tables = {i: pd.read_csv(RESULTS / i / "metrics.csv") for i in corpus.indexes()}
COLS = ["method", "gate", "budget", "section_ndcg@10", "section_ndcg@10_ci_lo", "section_ndcg@10_ci_hi",
        "section_recall@10", "section_mrr@10", "evidence_recall@10", "unit_ndcg@10", "latency_total_s"]

def rq_table(methods, index, params_version="v1", extra=()):
    t = tables[index]
    t = t[t["method"].isin(methods) & (t["params_version"] == params_version)]
    cols = [c for c in COLS + list(extra) if c in t.columns]
    return t[cols].sort_values(["method", "gate", "budget"]).round(3)

RQ = {
    "RQ1 single signals": ["L1", "S1", "D1", "C1"],
    "RQ2 hybrid and multi-stage": ["L1", "D1", "H1", "H2", "M1"],
    "RQ3 rerankers": ["H1", "X1", "X2"],
    "RQ4 LLM query rewrites": ["L1", "D1", "Q1", "Q2"],
    "RQ5 metadata and structure": ["H1", "F0", "F1", "T1"],
}
for name, methods in RQ.items():
    for index in corpus.indexes():
        display(Markdown(f"### {name} · `{index}`"))
        display(rq_table(methods, index))

In [ ]:
for index in corpus.indexes():
    display(Markdown(f"### RQ6 expansion · `{index}`"))
    display(rq_table(EXPANSION_METHODS, index, extra=("evidence_recall@ctx", "context_tokens", "llm_calls")))

In [ ]:
display(Markdown("### RQ7 windows vs units (gate off), section level and evidence recall"))
k7 = sig7[sig7["metric"].isin(["ndcg@10", "recall@10", "mrr@10", "evidence_recall@10", "evidence_recall@ctx"])]
k7[["a", "b", "params_version", "level", "metric", "mean_a", "mean_b", "diff", "diff_ci_lo", "diff_ci_hi", "p_holm"]].round(4)

## Plots

Colours: the reference categorical slots in fixed order (slot 1 blue, slot 2 orange, …). Every multi-series plot has a legend, so identity never depends on colour alone.

In [ ]:
import matplotlib.pyplot as plt

INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 10, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 6,
})

def metric(run_ids, level, name, field="value"):
    r = rm[(rm["subset"] == "all") & (rm["level"] == level) & (rm["metric"] == name)].set_index("run_id")[field]
    return r.reindex(run_ids)

### Section nDCG@10 by method, gate off vs on (v1)

95% bootstrap CIs; one panel per index, same x-axis.

In [ ]:
v1 = rt[rt["params_version"] == "v1"]
methods = [m for m in RANKED_METHODS if m in set(v1["method"])]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6), sharex=True, sharey=True)
for ax, index in zip(axes, corpus.indexes()):
    for j, (gate, label) in enumerate([(False, "gate off"), (True, "gate on")]):
        r = v1[(v1["index"] == index) & (v1["gate"] == gate) & v1["method"].isin(methods)].set_index("method").reindex(methods)
        y = np.arange(len(methods)) + (0.17 if j else -0.17)
        val = metric(r["run_id"], "section", "ndcg@10").values
        lo = metric(r["run_id"], "section", "ndcg@10", "ci_lo").values
        hi = metric(r["run_id"], "section", "ndcg@10", "ci_hi").values
        ax.hlines(y, lo, hi, color=SLOTS[j], linewidth=1.2)
        ax.plot(val, y, "o", color=SLOTS[j], label=label)
    ax.set_yticks(np.arange(len(methods)), methods)
    ax.invert_yaxis()
    ax.set_title(f"{index}")
    ax.set_xlabel("section nDCG@10 (95% CI)")
fig.suptitle("Ranked methods · n = 92 answerable", x=0.01, ha="left", color=INK)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncols=2)
fig.tight_layout(rect=(0, 0, 1, 0.95))

### RQ6: evidence recall within the budget vs context handed over (gate on, v1)

Each line is one expansion method across the 4k / 8k / 16k budgets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for ax, index in zip(axes, corpus.indexes()):
    for j, m in enumerate(EXPANSION_METHODS):
        r = v1[(v1["index"] == index) & v1["gate"] & (v1["method"] == m)].sort_values("budget")
        if r.empty:
            continue
        x = costs.set_index("run_id").loc[r["run_id"], "context_tokens"].values
        y = metric(r["run_id"], "evidence", "evidence_recall@ctx").values
        ax.plot(x, y, "-o", color=SLOTS[j], label=m)
    ax.set_title(index)
    ax.set_xlabel("mean context tokens per question")
axes[0].set_ylabel("evidence recall (whole context)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncols=5)
fig.tight_layout(rect=(0, 0, 1, 0.92))

### RQ7: windows vs units (gate off, v1)

In [ ]:
m7 = ["L1", "D1", "C1", "H1", "X1"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (level, name, title) in zip(axes, [("section", "ndcg@10", "section nDCG@10"), ("evidence", "evidence_recall@10", "evidence recall@10")]):
    for j, index in enumerate(corpus.indexes()):
        r = v1[(v1["index"] == index) & ~v1["gate"] & v1["method"].isin(m7)].set_index("method").reindex(m7)
        ax.plot(metric(r["run_id"], level, name).values, np.arange(len(m7)), "o", color=SLOTS[j], label=index, zorder=3)
    u = v1[(v1["index"] == "units") & ~v1["gate"] & v1["method"].isin(m7)].set_index("method").reindex(m7)["run_id"]
    w = v1[(v1["index"] == "win500") & ~v1["gate"] & v1["method"].isin(m7)].set_index("method").reindex(m7)["run_id"]
    ax.hlines(np.arange(len(m7)), metric(u, level, name).values, metric(w, level, name).values, color=GRID, linewidth=2, zorder=2)
    ax.set_yticks(np.arange(len(m7)), m7)
    ax.invert_yaxis()
    ax.set_title(title)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncols=2)
fig.tight_layout(rect=(0, 0, 1, 0.9))

### RQ7 by gold-section length: windows minus units, section nDCG@10 (gate off, v1)

Windows should differ most from base units on long sections.

In [ ]:
qf = rp._question_frame(questions, corpus).set_index("question_id")
sec = pq[(pq["level"] == "section") & (pq["metric"] == "ndcg@10")].join(qf["length_bucket"], on="question_id")
buckets = ["≤500", "501–1500", ">1500"]
fig, ax = plt.subplots(figsize=(7, 3.2))
width = 0.16
for j, m in enumerate(m7):
    d = []
    for b in buckets:
        ids = {i: v1[(v1["index"] == i) & ~v1["gate"] & (v1["method"] == m)]["run_id"] for i in corpus.indexes()}
        if any(x.empty for x in ids.values()):
            d.append(np.nan)
            continue
        s = {i: sec[(sec["run_id"] == ids[i].iloc[0]) & (sec["length_bucket"] == b)].set_index("question_id")["value"] for i in ids}
        d.append(float((s["win500"] - s["units"]).mean()))
    ax.bar(np.arange(len(buckets)) + (j - 2) * width, d, width * 0.9, color=SLOTS[j], label=m)
ax.axhline(0, color=INK2, linewidth=0.8)
ax.set_xticks(np.arange(len(buckets)), [f"{b} tokens\n(n = {int((qf['length_bucket'] == b).sum())})" for b in buckets])
ax.set_ylabel("Δ section nDCG@10 (win500 − units)")
ax.legend(ncols=5, loc="upper left", bbox_to_anchor=(0, 1.15))
fig.tight_layout()

## Gate diagnostics

Kept length and gate loss per gate position (gate A per signal, gate B after reranking), and the overall gate loss against the gate-off run of the same configuration.

In [ ]:
g = gates.merge(rt[["run_id", "method", "index", "budget", "params_version"]], on="run_id")
g = g.merge(gate_overall, on="run_id", how="left")
g.sort_values(["params_version", "index", "method", "position"])[["method", "index", "budget", "position", "in_mean", "kept_mean",
                                                                  "kept_median", "gate_loss", "gate_loss_overall"]].round(3)

## Agent decision paths (T4)

Per question: `stop` (the hits were enough), `expand`, `follow_refs`, `search`, `limit` (step limit reached).

In [ ]:
rows = []
for rid in rt[rt["method"] == "T4"]["run_id"]:
    c = runs.load_config(rid)
    for s in runs.load_stats(rid):
        p = s["agent"]["path"]
        kind = "stop at once" if p == ["stop"] else ("step limit" if "limit" in p else "+".join(sorted({x for x in p if x != "stop"})))
        rows.append({"index": c["index"], "budget": c["config"]["expansion"]["budget"], "params": c["params_version"], "path": kind})
if rows:
    display(pd.DataFrame(rows).pivot_table(index="path", columns=["params", "index", "budget"], aggfunc="size", fill_value=0))

## Cost per question

In [ ]:
costs.merge(rt, on="run_id").groupby(["params_version", "index", "method"])[
    ["latency_qdrant_s", "latency_encoder_s", "latency_rerank_s", "latency_llm_s", "latency_total_s",
     "llm_prompt_tokens", "llm_completion_tokens", "context_tokens"]].mean().round(2)

## Unanswerable questions (inspection only, not scored)

In [ ]:
un = pd.read_excel(RESULTS / "units" / "metrics.xlsx", sheet_name="unanswerable")
un[un["run_id"].str.startswith(("H1-", "X1-", "X2-"))][["run_id", "question_id", "top5_sections", "n_distractors_in_top5", "top1_score", "n_hits"]].head(24)

## Pipeline diagrams

Picked by hand for the write-up (here: one per RQ). Each is generated from the run's stored config and written to `results/diagrams/<run_id>.md`.

In [ ]:
def pick(method, index, gate, budget=None, params_version="v1"):
    r = rt[(rt["method"] == method) & (rt["index"] == index) & (rt["gate"] == gate) & (rt["params_version"] == params_version)]
    if budget is not None:
        r = r[r["budget"] == budget]
    return r["run_id"].iloc[0] if len(r) else None

DIAGRAM_RUNS = [x for x in [
    pick("C1", "units", False),            # RQ1
    pick("H1", "units", True),             # RQ2
    pick("X2", "units", True),             # RQ3
    pick("Q1", "units", False),            # RQ4
    pick("F1", "units", True),             # RQ5
    pick("T4", "units", True, 16384),      # RQ6
    pick("H1", "win500", False),           # RQ7
] if x]
for rid in DIAGRAM_RUNS:
    path = rp.write_diagram(rid, RESULTS / "diagrams", rm=rm, sig=pd.concat([sig, sig7]), gates=gates, costs=costs)
    display(Markdown(path.read_text(encoding="utf-8")))

## Output checks (plan 8)

* both `metrics.csv` / `metrics.xlsx` pairs and `granularity.csv` match `run_metrics.parquet` (same run count and values);
* each CSV opens cleanly: UTF-8, one header row;
* the `unanswerable` sheet lists all 8 questions for every run;
* each diagram's stage list matches its run's `config.json`.

In [ ]:
nd = rm[(rm["subset"] == "all") & (rm["level"] == "section") & (rm["metric"] == "ndcg@10")].set_index("run_id")["value"]
for index in corpus.indexes():
    t = pd.read_csv(RESULTS / index / "metrics.csv", encoding="utf-8")
    assert len(t) == (rt["index"] == index).sum(), "run count differs"
    assert np.allclose(t.set_index("run_id")["section_ndcg@10"], nd.reindex(t["run_id"]).values, equal_nan=True)
    with open(RESULTS / index / "metrics.csv", encoding="utf-8") as f:
        header = f.readline()
        assert "run_id" in header and all("run_id" not in ln for ln in f), "more than one header row"
    x = pd.ExcelFile(RESULTS / index / "metrics.xlsx")
    assert {"metrics", "by_question_type", "by_engine_type", "by_kind", "mentions_section_id", "length_bucket",
            "significance", "gate", "per_question", "unanswerable"} <= set(x.sheet_names)
    xm = pd.read_excel(x, "metrics")
    assert np.allclose(xm.set_index("run_id")["section_ndcg@10"], nd.reindex(xm["run_id"]).values, equal_nan=True)
    un = pd.read_excel(x, "unanswerable")
    assert un.groupby("run_id")["question_id"].nunique().eq(8).all() and un["run_id"].nunique() == len(t)
g7 = pd.read_csv(RESULTS / "granularity.csv")
assert len(g7) == len(sig7)
assert all(rp.diagram_matches_config(r) for r in DIAGRAM_RUNS)
print("output checks passed")